In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
!pip install ultralytics roboflow -q

In [ ]:
# Mount Google Drive so runs/weights are saved and survive session resets
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Download dataset from Roboflow
# Dataset: Fire and Smoke Detection (public, ~6k images, 2 classes: fire, smoke)
# https://universe.roboflow.com/momoney/fire-and-smoke-qcvzt

from roboflow import Roboflow
import os

# Get your free API key at https://roboflow.com -> top right -> API key
RF_API_KEY = 'YOUR_ROBOFLOW_API_KEY'  # <-- paste your key here

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace('momoney').project('fire-and-smoke-qcvzt')
dataset = project.version(4).download('yolov8', location='/content/dataset')

DATA_YAML = dataset.location + '/data.yaml'
print('data.yaml:', DATA_YAML)
print('exists:', os.path.exists(DATA_YAML))

In [ ]:
# Preview data.yaml
with open(DATA_YAML, 'r') as f:
    print(f.read())

In [ ]:
# Peek at some training images
import glob, random, matplotlib.pyplot as plt
import cv2

imgs = glob.glob('/content/dataset/train/images/*.jpg')
sample = random.sample(imgs, min(6, len(imgs)))

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, path in zip(axes.flatten(), sample):
    ax.imshow(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'Total train images: {len(imgs)}')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # n=nano (fast), swap to yolov8s/m/l for more accuracy

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    name='fire_smoke_v1',
    project='/content/drive/MyDrive/runs',  # saves to Drive
    patience=10,
    device=0
)

In [ ]:
# Show training curves
from IPython.display import Image
Image('/content/drive/MyDrive/runs/fire_smoke_v1/results.png')

In [ ]:
# Validate with best weights
best = YOLO('/content/drive/MyDrive/runs/fire_smoke_v1/weights/best.pt')
best.val(data=DATA_YAML, device=0)

In [ ]:
# Run inference on val images
best.predict(
    source='/content/dataset/valid/images',
    conf=0.25,
    save=True,
    project='/content/drive/MyDrive/runs',
    name='test_output'
)

In [ ]:
# Show some prediction results
import glob, random, matplotlib.pyplot as plt
import cv2

pred_imgs = glob.glob('/content/drive/MyDrive/runs/test_output/*.jpg')
sample = random.sample(pred_imgs, min(6, len(pred_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, path in zip(axes.flatten(), sample):
    ax.imshow(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Export to ONNX (optional, for deployment)
best.export(format='onnx')